# TDA Supply Chain Anomaly Detection — Interactive Notebook

This notebook demonstrates:
1. Building a dynamic supply-chain simplicial complex
2. Computing persistent homology (H₀, H₁, H₂)
3. Wasserstein-distance anomaly scoring
4. Visualizing persistence diagrams, barcodes, and landscapes
5. Running a full simulation with known disruptions

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import time

from src.core.tda_engine import (
    DynamicFilteredComplex, PersistentHomologyComputer,
    wasserstein_distance, multi_dim_wasserstein, persistence_landscape
)
from src.core.anomaly_detector import TopologicalAnomalyDetector
from src.streaming.pipeline import StreamingPipeline, ContainerEvent
from src.data_gen.synthetic_generator import SyntheticDataGenerator
from src.visualization.visualizer import (
    plot_persistence_diagram, plot_barcode, plot_landscape, plot_anomaly_timeline,
    export_html_heatmap
)

print('All imports successful ✓')

## 1. Build a minimal supply-chain complex
Start with a small graph: 5 ports forming a known topology.

In [ ]:
# Build a graph with 5 ports
cplx = DynamicFilteredComplex()

# Add ports (nodes)
for i in range(5):
    cplx.update_node(i, weight=0.0)

# Add trade routes (edges) with filtration weights representing congestion
routes = [
    (0, 1, 0.1),   # Shanghai → Singapore  (healthy)
    (1, 2, 0.15),  # Singapore → Rotterdam (healthy)
    (2, 3, 0.2),   # Rotterdam → New York  (slight delay)
    (3, 4, 0.1),   # New York  → LA        (healthy)
    (0, 2, 0.25),  # Shanghai → Rotterdam  (direct, moderate)
]

for u, v, w in routes:
    cplx.update_edge(u, v, weight=w)

print(f'Complex: {cplx.n_nodes} nodes, {cplx.n_edges} edges, {cplx.n_simplices} total simplices')

## 2. Compute Persistent Homology

In [ ]:
comp = PersistentHomologyComputer()
diagram = comp.update(cplx)

print(f'Betti numbers: β₀={diagram.betti_numbers.get(0,0)}, '
      f'β₁={diagram.betti_numbers.get(1,0)}, '
      f'β₂={diagram.betti_numbers.get(2,0)}')
print(f'Total persistence pairs: {len(diagram.pairs)}')

print('\nH1 pairs (routing cycles):')
for p in diagram.pairs_by_dim(1):
    death_str = f'{p.death:.3f}' if not np.isinf(p.death) else '∞'
    print(f'  birth={p.birth:.3f}, death={death_str}, persistence={p.persistence if not np.isinf(p.persistence) else "∞":.3f}')

## 3. Visualize Persistence Diagram and Barcode

In [ ]:
fig = plot_persistence_diagram(diagram, title='Supply Chain — Normal Operation')
plt.tight_layout()
plt.show()

fig2 = plot_barcode(diagram)
if fig2:
    plt.tight_layout()
    plt.show()

## 4. Simulate a Suez Canal Blockage

In [ ]:
# Remove the key transit edge (simulating a canal blockage)
cplx_disrupted = DynamicFilteredComplex()
for i in range(5):
    cplx_disrupted.update_node(i, weight=0.0)

disrupted_routes = [
    (0, 1, 0.1),
    (1, 2, 0.85),   # Suez gateway: massive congestion
    (2, 3, 0.75),   # Downstream knock-on
    (3, 4, 0.1),
    (0, 2, 0.9),    # Rerouting attempt — high load
    (0, 4, 0.7),    # Emergency route
    (1, 3, 0.8),    # Another emergency route
]
for u, v, w in disrupted_routes:
    cplx_disrupted.update_edge(u, v, weight=w)

diagram_disrupted = comp.update(cplx_disrupted)

print('DISRUPTED Betti numbers:')
print(f'  β₀={diagram_disrupted.betti_numbers.get(0,0)}, '
      f'β₁={diagram_disrupted.betti_numbers.get(1,0)}, '
      f'β₂={diagram_disrupted.betti_numbers.get(2,0)}')

# Compute Wasserstein distance
wd = multi_dim_wasserstein(diagram, diagram_disrupted)
print(f'\nWasserstein distance (normal → disrupted): {wd:.4f}')

## 5. Persistence Landscape

In [ ]:
fig3 = plot_landscape(diagram_disrupted, dim=1, n_layers=3)
if fig3:
    plt.tight_layout()
    plt.show()

## 6. Full Streaming Simulation

In [ ]:
gen = SyntheticDataGenerator(n_nodes=30, seed=42)
pipe = StreamingPipeline(tick_seconds=1e-9, window_size=20, warmup_steps=15)

n_normal    = 20
n_disrupted = 25
scenario    = 'suez_blockage'

scores   = []
labels   = []
alerts   = []

for step in range(n_normal + n_disrupted):
    disrupted = step >= n_normal
    events = gen.generate_step(step, disrupted=disrupted, scenario=scenario)
    last_alert = None
    for e in events:
        last_alert = pipe.process_sync(e)
    if last_alert:
        scores.append(last_alert.anomaly_score)
        labels.append(int(disrupted))
        alerts.append(last_alert)
        icon = '🔴' if last_alert.is_anomaly else '🟢'
        phase = 'DISRUPTED' if disrupted else 'normal'
        print(f'Step {step:3d} [{phase:9s}] {icon}  score={last_alert.anomaly_score:.4f}'
              f'  B=({last_alert.betti_delta})')

print(f'\nPipeline: {pipe.complex.n_nodes} nodes, {pipe.complex.n_edges} edges')

## 7. Anomaly Timeline Visualization

In [ ]:
fig4 = plot_anomaly_timeline(alerts)
if fig4:
    plt.tight_layout()
    plt.show()

# Quick detection metrics
n_tp = sum(1 for a, l in zip(alerts, labels) if a.is_anomaly and l == 1)
n_fp = sum(1 for a, l in zip(alerts, labels) if a.is_anomaly and l == 0)
n_pos = sum(labels)
n_neg = len(labels) - n_pos
tpr = n_tp / n_pos if n_pos else 0
fpr = n_fp / n_neg if n_neg else 0
print(f'TPR: {tpr:.1%}  |  FPR: {fpr:.1%} (target < 1%)')

## 8. Export Interactive HTML Heatmap

In [ ]:
export_html_heatmap(
    pipe.complex,
    pipe._latest_alert,
    pipe.latest_diagram,
    output_path='supply_chain_heatmap.html'
)
print('Open supply_chain_heatmap.html in your browser to explore the interactive network.')